# Experiment: First Wet-Lab Panel Optimizer

Objective: choose the smallest qualified-lab pilot panel that covers the current Nakafa Lab cure-oriented readouts without pretending that a biological experiment has already been run.

Success criteria:
- include a positive HbF comparator;
- include at least one HbF seed to test against HPFH-like readouts;
- include alpha-globin cleanup or autophagy biology;
- include red-cell metabolism or mature red-cell safety;
- include a hazard boundary so unsafe materials are rejected early;
- keep the panel small enough for a first external lab quote.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

REQUIRED_TAGS = {
    "positive_hbf_control",
    "hbf_seed_for_hpfh_like_readouts",
    "autophagy_alpha_cleanup",
    "red_cell_metabolism_or_support",
    "hemolysis_hazard_boundary",
}

MAX_PANEL_SIZE = 6

## Plan

This is a computational design experiment, not wet-lab evidence.

Hypothesis: a first pilot panel can be smaller than the full work order if each candidate is selected for coverage of a required biological question.

Metric: maximize required-tag coverage first, then prefer higher research priority and lower practical friction.


In [ ]:
@dataclass(frozen=True)
class Candidate:
    """One candidate or comparator for first-pass lab-panel design."""

    name: str
    role: str
    priority: float
    friction: float
    tags: frozenset[str]
    boundary: str

    @property
    def utility(self) -> float:
        """Return a simple priority-minus-friction utility score."""
        return self.priority - self.friction

In [ ]:
candidates = [
    Candidate(
        name="hydroxyurea",
        role="positive HbF comparator",
        priority=5.0,
        friction=1.0,
        tags=frozenset(
            {"positive_hbf_control", "hbf_readout", "local_access_comparator"}
        ),
        boundary="clinician-supervised comparator, not a new Nakafa Lab cure",
    ),
    Candidate(
        name="sirolimus",
        role="mTOR/autophagy comparator",
        priority=4.0,
        friction=3.0,
        tags=frozenset({"hbf_readout", "autophagy_alpha_cleanup"}),
        boundary="immune-active comparator with BPOM public-search access gap",
    ),
    Candidate(
        name="T-BDMC-like curcuminoid analog",
        role="natural-product-adjacent HbF seed",
        priority=3.5,
        friction=2.5,
        tags=frozenset(
            {"hbf_seed_for_hpfh_like_readouts", "hbf_readout", "identity_gate"}
        ),
        boundary="requires exact compound identity and hemolysis gate",
    ),
    Candidate(
        name="resveratrol purified compound",
        role="natural-product HbF seed",
        priority=3.0,
        friction=1.5,
        tags=frozenset({"hbf_seed_for_hpfh_like_readouts", "hbf_readout"}),
        boundary="seed signal only; needs endogenous HbF and safety replication",
    ),
    Candidate(
        name="mitapivat or lab-approved PK activator reference",
        role="red-cell metabolism benchmark",
        priority=3.5,
        friction=3.5,
        tags=frozenset({"red_cell_metabolism_or_support", "hemolysis_readout"}),
        boundary="include only if legally and practically available to the lab",
    ),
    Candidate(
        name="standardized 6-shogaol-rich ginger extract",
        role="red-cell support comparator",
        priority=2.0,
        friction=1.5,
        tags=frozenset({"red_cell_metabolism_or_support", "hemolysis_readout"}),
        boundary="support endpoint only, not an HbF or cure claim",
    ),
    Candidate(
        name="melittin hazard control",
        role="hemolysis hazard boundary",
        priority=1.0,
        friction=1.0,
        tags=frozenset({"hemolysis_hazard_boundary"}),
        boundary="hazard calibration only; not a therapy candidate",
    ),
]

len(candidates)

In [ ]:
def choose_panel(pool: list[Candidate]) -> list[Candidate]:
    """Choose a small panel by greedily covering required assay tags."""
    selected: list[Candidate] = []
    covered: set[str] = set()
    remaining = pool.copy()

    while remaining and len(selected) < MAX_PANEL_SIZE:
        ranked = sorted(
            remaining,
            key=lambda item: (
                len((item.tags & REQUIRED_TAGS) - covered),
                item.utility,
                item.priority,
            ),
            reverse=True,
        )
        best = ranked[0]
        new_tags = (best.tags & REQUIRED_TAGS) - covered
        if not new_tags and covered >= REQUIRED_TAGS:
            break
        selected.append(best)
        covered.update(best.tags & REQUIRED_TAGS)
        remaining.remove(best)

        if covered >= REQUIRED_TAGS:
            break

    return selected


panel = choose_panel(candidates)
covered_tags = sorted(set().union(*(item.tags for item in panel)) & REQUIRED_TAGS)
missing_tags = sorted(REQUIRED_TAGS - set(covered_tags))
(len(panel), covered_tags, missing_tags)

In [ ]:
panel_summary = [
    {
        "candidate": item.name,
        "role": item.role,
        "utility": round(item.utility, 2),
        "covered_required_tags": sorted(item.tags & REQUIRED_TAGS),
        "boundary": item.boundary,
    }
    for item in panel
]

panel_summary

## Results

The expected first-pass output is not a cure claim. It is a focused lab quote package.

Interpretation rule: promote nothing unless the later wet-lab data show HbF or support readouts without viability, maturation, or hemolysis failure.


In [ ]:
decision = {
    "panel_size": len(panel),
    "required_tags_covered": covered_tags,
    "required_tags_missing": missing_tags,
    "decision": "ready_for_lab_quote" if not missing_tags else "needs_more_candidates",
}

decision

## Next steps

- Convert the selected panel into a one-page lab outreach table.
- Keep patient samples out of scope until ethics approval, consent, and clinician oversight exist.
- Record any lab quote constraints back into the assay work order.
